In [ ]:
# 1
from google.colab import drive
import os, re, json, math, shutil, subprocess, hashlib, sys

drive.mount('/content/drive', force_remount=False)

FOLDER='/content/drive/MyDrive/AIgenerated'
os.makedirs(FOLDER,exist_ok=True)
SONG_PATH=os.path.join(FOLDER,'song.mp3')
LYRICS_PATH=os.path.join(FOLDER,'lyrics.txt')
TRIMMED_AUDIO_PATH=os.path.join(FOLDER,'trimmed_audio.m4a')
SONG_MAP_PATH=os.path.join(FOLDER,'song_map.json')

# USER INPUT — approximate search window only.
SEARCH_START=0.0
SEARCH_END=40.0

print('✅ Workspace ready')
print('Song:',SONG_PATH)
print('Lyrics:',LYRICS_PATH)
print(f'SEARCH window: {SEARCH_START:.3f}s -> {SEARCH_END:.3f}s')
print('⚠️ This window is NOT the final audio duration.')

In [ ]:
# 2
if not os.path.isfile(SONG_PATH):
    raise FileNotFoundError(f'Missing song.mp3: {SONG_PATH}')
if not os.path.isfile(LYRICS_PATH):
    raise FileNotFoundError(f'Missing lyrics.txt: {LYRICS_PATH}')

with open(LYRICS_PATH,'r',encoding='utf-8') as f:
    LYRICS_LINES=[x.strip() for x in f if x.strip()]

if not LYRICS_LINES:
    raise RuntimeError('lyrics.txt has no usable lyric lines.')

probe=subprocess.run([
    'ffprobe','-v','error','-show_entries','format=duration',
    '-of','default=noprint_wrappers=1:nokey=1',SONG_PATH
],capture_output=True,text=True,check=True)
SONG_DURATION=float(probe.stdout.strip())

if SEARCH_START<0 or SEARCH_START>=SONG_DURATION:
    raise ValueError(f'Invalid SEARCH_START: {SEARCH_START}')
SEARCH_END=min(float(SEARCH_END),SONG_DURATION)
if SEARCH_END<=SEARCH_START:
    raise ValueError('SEARCH_END must be greater than SEARCH_START.')

print('INPUT VALIDATION')
print('================')
print(f'Song duration : {SONG_DURATION:.3f}s')
print(f'Lyrics lines  : {len(LYRICS_LINES)}')
print(f'Search window : {SEARCH_START:.3f}s -> {SEARCH_END:.3f}s')
for i,x in enumerate(LYRICS_LINES,1): print(f'{i:02d}. {x}')
print('✅ Inputs valid')

In [ ]:
# 3
# ONLY an analysis window. It is never treated as final audio.
SEARCH_AUDIO='/tmp/cutie_stage1_search.m4a'

subprocess.run([
    'ffmpeg','-y','-ss',f'{SEARCH_START:.3f}','-i',SONG_PATH,
    '-t',f'{SEARCH_END-SEARCH_START:.3f}',
    '-vn','-c:a','aac','-b:a','192k',SEARCH_AUDIO
],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE)

print(f'✅ Temporary acoustic window: {SEARCH_AUDIO}')
print(f'Window duration: {SEARCH_END-SEARCH_START:.3f}s')
print('No final audio is created here.')

In [ ]:
# 4
try:
    from faster_whisper import WhisperModel
except ModuleNotFoundError:
    subprocess.run([sys.executable,'-m','pip','install','-q','faster-whisper'],check=True)
    from faster_whisper import WhisperModel

print('Loading multilingual Whisper model...')
model=WhisperModel('small',device='cuda',compute_type='float16')

print('Transcribing search window for timing evidence...')
segments,info=model.transcribe(
    SEARCH_AUDIO,
    language='hi',
    word_timestamps=True,
    vad_filter=False,
    condition_on_previous_text=True,
    beam_size=5,
    best_of=5
)
segments=list(segments)

EVIDENCE=[]
for seg in segments:
    for w in (seg.words or []):
        text=(w.word or '').strip()
        if not text or w.start is None or w.end is None: continue
        ls=float(w.start); le=float(w.end)
        if le<=ls: continue
        EVIDENCE.append({
            'start':round(SEARCH_START+ls,3),
            'end':round(SEARCH_START+le,3),
            'text':text
        })

if not EVIDENCE:
    raise RuntimeError('No usable Whisper word timing evidence found.')

print('Detected language:',info.language)
print('Model segments:',len(segments))
print('Evidence words:',len(EVIDENCE))
print('\nMODEL TIMING EVIDENCE (timing only):')
for i,w in enumerate(EVIDENCE,1):
    print(f"{i:03d}. {w['start']:.3f} -> {w['end']:.3f} : {w['text']}")

In [ ]:

# IMPORTANT: lyrics.txt is authoritative. Whisper words are evidence only.
# This cell aligns the COMPLETE supplied lyric sequence in chronological order.
# It never stretches the last line to SEARCH_END.

subprocess.run([sys.executable,'-m','pip','install','-q','indic-transliteration','rapidfuzz'],check=True)
from indic_transliteration import sanscript
from rapidfuzz.fuzz import ratio
import unicodedata

WORD_RE=re.compile(r'[A-Za-z0-9\u0900-\u097f]+',re.UNICODE)

def clean(s):
    s=unicodedata.normalize('NFKC',str(s)).lower()
    return re.sub(r'[^a-z0-9\u0900-\u097f]+',' ',s).strip()

def phonetic(s):
    s=clean(s)
    if re.search(r'[\u0900-\u097f]',s):
        try: s=sanscript.transliterate(s,sanscript.DEVANAGARI,sanscript.ITRANS)
        except Exception: pass
    s=s.lower()
    for a,b in [('aa','a'),('ee','i'),('ii','i'),('oo','u'),('ou','u'),('kh','k'),('gh','g'),('chh','ch'),('jh','j'),('sh','s'),('ph','f'),('bh','b'),('th','t'),('dh','d'),('v','w')]: s=s.replace(a,b)
    return re.sub(r'[^a-z0-9]','',s)

def sim(a,b):
    a=phonetic(a); b=phonetic(b)
    if not a or not b: return 0.0
    if a==b: return 1.0
    if a in b or b in a: return 0.88
    return ratio(a,b)/100.0

def display_words(line):
    return WORD_RE.findall(line)

# Flatten supplied lyrics while retaining line ownership.
LYRIC_WORDS=[]
for line_no,line in enumerate(LYRICS_LINES,1):
    for word in display_words(line):
        LYRIC_WORDS.append({'line_no':line_no,'word':word})

MODEL_WORDS=EVIDENCE

# Dynamic-programming alignment over the full sequence.
# This is intentionally more conservative than independent per-line matching.
N=len(LYRIC_WORDS); M=len(MODEL_WORDS)
if N==0: raise RuntimeError('No lyric words found.')

# Candidate match score. A skipped model word is cheap; a lyric word can be
# unmatched and later interpolated. Strong lexical similarity is rewarded.
SKIP_MODEL=-0.12
SKIP_LYRIC=-0.55
MATCH_FLOOR=0.48

# Limit candidates so repeated words do not jump across the whole search window.
from functools import lru_cache

@lru_cache(maxsize=None)
def dp(i,j):
    if i>=N: return 0.0
    if j>=M: return SKIP_LYRIC*(N-i)
    best=SKIP_MODEL+dp(i,j+1)
    best=max(best,SKIP_LYRIC+dp(i+1,j))
    s=sim(LYRIC_WORDS[i]['word'],MODEL_WORDS[j]['text'])
    # Matching becomes less attractive when a large amount of audio time was
    # skipped between consecutive matched words.
    if s>=MATCH_FLOOR:
        gap=0.0
        if i>0:
            # Find prior model position indirectly through the DP state is hard;
            # therefore sequence order is handled here and time gaps are checked
            # again during reconstruction.
            gap=0.0
        best=max(best,s+dp(i+1,j+1))
    return best

# Reconstruct the highest-scoring monotonic sequence.
matches=[]; i=0; j=0
while i<N and j<M:
    current=dp(i,j)
    skip_m=SKIP_MODEL+dp(i,j+1)
    skip_l=SKIP_LYRIC+dp(i+1,j)
    s=sim(LYRIC_WORDS[i]['word'],MODEL_WORDS[j]['text'])
    take=(s>=MATCH_FLOOR and abs(current-(s+dp(i+1,j+1)))<1e-8)
    if take:
        matches.append({'lyric_i':i,'model_i':j,'score':round(s,3),'word':LYRIC_WORDS[i]['word'],'line_no':LYRIC_WORDS[i]['line_no'],'model':MODEL_WORDS[j]})
        i+=1; j+=1
    elif skip_m>=skip_l:
        j+=1
    else:
        i+=1

# Remove implausible timing leaps between accepted anchors. If a jump is too
# large, the later anchor is rejected and the missing lyric words are interpolated.
MAX_ANCHOR_GAP=4.0
safe=[]
for a in matches:
    if not safe or a['model']['start']-safe[-1]['model']['end']<=MAX_ANCHOR_GAP:
        safe.append(a)
    else:
        # Keep only a strong anchor across a larger gap; weak accidental matches
        # are discarded so they cannot create a 10–20 second line.
        if a['score']>=0.92 and a['line_no']>safe[-1]['line_no']:
            safe.append(a)

matches=safe

# Group accepted anchors by lyric line.
line_anchors={i:[] for i in range(1,len(LYRICS_LINES)+1)}
for a in matches: line_anchors[a['line_no']].append(a)

# A line can only use anchors belonging to itself. Determine boundaries from
# its anchors, then interpolate unanchored lines between neighboring lines.
LINE_DATA=[]
for line_no,line in enumerate(LYRICS_LINES,1):
    aa=line_anchors[line_no]
    if aa:
        s=aa[0]['model']['start']; e=aa[-1]['model']['end']
        LINE_DATA.append({'line_no':line_no,'line':line,'start':s,'end':e,'anchored':True,'anchors':aa})
    else:
        LINE_DATA.append({'line_no':line_no,'line':line,'start':None,'end':None,'anchored':False,'anchors':[]})

known=[i for i,x in enumerate(LINE_DATA) if x['anchored']]
if not known: raise RuntimeError('No reliable lyric/audio anchors found. Increase SEARCH window or check lyrics.')

# Fill missing line ranges without ever using SEARCH_END as an artificial lyric end.
for idx,x in enumerate(LINE_DATA):
    if x['anchored']: continue
    prev=max([k for k in known if k<idx],default=None)
    nxt=min([k for k in known if k>idx],default=None)
    if prev is not None and nxt is not None:
        left=LINE_DATA[prev]['end']; right=LINE_DATA[nxt]['start']
        count=nxt-prev
        x['start']=left+(right-left)*(idx-prev)/count
        x['end']=left+(right-left)*(idx-prev+1)/count
    elif prev is not None:
        # Estimate only a modest tail from the last anchor; never to SEARCH_END.
        left=LINE_DATA[prev]['end']
        next_anchor=next((a for a in matches if a['line_no']>idx+1),None)
        step=1.0
        if next_anchor: step=max(0.25,min(2.5,next_anchor['model']['start']-left))
        x['start']=left
        x['end']=left+step
    else:
        right=LINE_DATA[nxt]['start']
        step=max(0.25,min(2.5,(right-SEARCH_START)/max(1,nxt+1)))
        x['end']=right
        x['start']=right-step

# Enforce monotonic line timing.
for i,x in enumerate(LINE_DATA):
    x['start']=float(x['start']); x['end']=float(x['end'])
    if i>0: x['start']=max(x['start'],LINE_DATA[i-1]['end'])
    if x['end']<=x['start']: x['end']=x['start']+0.05
    x['start']=round(x['start'],3); x['end']=round(x['end'],3); x['duration']=round(x['end']-x['start'],3)

print('SONG MAP PREVIEW')
print('=================')
for x in LINE_DATA:
    print(f"{x['line_no']:02d}. {x['start']:.3f}s -> {x['end']:.3f}s ({x['duration']:.3f}s) | {x['line']}")

# Build word-level timing. Anchored words retain audio timing; gaps are interpolated.
FINAL_LINES=[]
for line in LINE_DATA:
    words=display_words(line['line'])
    if not words: continue
    aa=sorted(line['anchors'],key=lambda z:z['lyric_i'])
    by_i={a['lyric_i']:a for a in aa}
    # lyric_i is global; find this line's local indices
    local=[]
    global_indices=[i for i,w in enumerate(LYRIC_WORDS) if w['line_no']==line['line_no']]
    for pos,gidx in enumerate(global_indices):
        a=by_i.get(gidx)
        local.append({'pos':pos,'word':LYRIC_WORDS[gidx]['word'],'anchor':a})
    # Sequentially interpolate between anchors.
    anchor_positions=[k for k,v in enumerate(local) if v['anchor'] is not None]
    for k,item in enumerate(local):
        if item['anchor'] is not None:
            ws=max(line['start'],item['anchor']['model']['start'])
            we=min(line['end'],max(ws+0.01,item['anchor']['model']['end']))
            item['start']=ws; item['end']=we; item['timing']='audio_anchor'
    for k,item in enumerate(local):
        if 'start' in item: continue
        prevs=[p for p in anchor_positions if p<k]
        nexts=[p for p in anchor_positions if p>k]
        if prevs:
            p=prevs[-1]; left=local[p]['end']
        else:
            p=None; left=line['start']
        if nexts:
            q=nexts[0]; right=local[q]['start']
            slots=q-(p if p is not None else -1)
            step=max(0.01,(right-left)/max(1,slots))
            item['start']=left+step*(k-(p if p is not None else -1)-1)
            item['end']=item['start']+step
        else:
            remaining=len(local)-k
            step=max(0.01,(line['end']-left)/max(1,remaining))
            item['start']=left
            item['end']=left+step
        item['timing']='interpolated'
    # Clamp/repair monotonicity.
    prev=line['start']
    for item in local:
        item['start']=max(float(item['start']),prev)
        item['end']=max(float(item['end']),item['start']+0.01)
        item['start']=min(item['start'],line['end']-0.01)
        item['end']=min(item['end'],line['end'])
        prev=item['end']
    FINAL_LINES.append({'line_no':line['line_no'],'line':line['line'],'start':line['start'],'end':line['end'],'duration':line['duration'],'words':[{'word':z['word'],'start':round(z['start'],3),'end':round(z['end'],3),'timing':z['timing']} for z in local]})

# Expose variables for Cell 6/7 compatibility.
final_lines=FINAL_LINES
print('\nWORD TIMING READY')
print('Total lines:',len(final_lines))
print('Total words:',sum(len(x['words']) for x in final_lines))

In [ ]:
# 6
import json, hashlib, os

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

with open(LYRICS_PATH,'rb') as f: lyrics_hash=hashlib.sha256(f.read()).hexdigest()
SONG_MAP={
    'version':3,
    'source_song':os.path.basename(SONG_PATH),
    'lyrics_file':os.path.basename(LYRICS_PATH),
    'lyrics_sha256':lyrics_hash,
    'search_window':{'start':SEARCH_START,'end':SEARCH_END},
    'lines':final_lines
}
with open(SONG_MAP_PATH,'w',encoding='utf-8') as f:
    json.dump(SONG_MAP,f,ensure_ascii=False,indent=2)
print(f'✅ Saved: {SONG_MAP_PATH}')
print(f"Calculated final source window: {final_lines[0]['start']:.3f}s -> {final_lines[-1]['end']:.3f}s")

In [ ]:
# 7
import subprocess, os

FINAL_START=max(0.0, float(final_lines[0]['start']) - 0.5)
FINAL_END=float(final_lines[-1]['end'])

if FINAL_END<=FINAL_START: raise RuntimeError('Invalid final lyric window.')

subprocess.run([
    'ffmpeg','-y','-ss',f'{FINAL_START:.3f}','-i',SONG_PATH,
    '-t',f'{FINAL_END-FINAL_START:.3f}',
    '-vn','-c:a','aac','-b:a','192k',TRIMMED_AUDIO_PATH
],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE)

# Remove only temporary search artifacts.
try: os.remove(SEARCH_AUDIO)
except FileNotFoundError: pass

print('=========================================')
print('✅ STAGE 1 COMPLETE')
print(f'Final audio: {TRIMMED_AUDIO_PATH}')
print(f'Duration: {FINAL_END-FINAL_START:.3f}s')
print(f'Song map: {SONG_MAP_PATH}')